In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from catboost import CatBoostClassifier
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier

warnings.filterwarnings('ignore')


COMPETITION_DATA_ROOT = '/kaggle/input/competitions/birdclef-2026'

FEATURE_DATASET_PATH = '/kaggle/input/datasets/shuoyiyang/birdclef-2026-features-v2-r3-t3'

print("Libraries imported and paths set.")

Libraries imported and paths set.


In [2]:
taxonomy_df = pd.read_csv(f'{COMPETITION_DATA_ROOT}/taxonomy.csv')
ALL_SPECIES = taxonomy_df['primary_label'].unique().tolist()

print(f"Total species defined: {len(ALL_SPECIES)}")

Total species defined: 234


In [3]:
INPUT_FEATURE_PATH = f'{FEATURE_DATASET_PATH}/features_v2_rating2_top3.parquet'
INPUT_LABEL_PATH   = f'{FEATURE_DATASET_PATH}/labels_v2_rating2_top3.parquet'

if os.path.exists(INPUT_FEATURE_PATH) and os.path.exists(INPUT_LABEL_PATH):
    print(f"Success! Loading features from: {INPUT_FEATURE_PATH}")
    X_df = pd.read_parquet(INPUT_FEATURE_PATH)
    valid_train_df = pd.read_parquet(INPUT_LABEL_PATH)
    
    mlb = MultiLabelBinarizer(classes=ALL_SPECIES)
    y_matrix = mlb.fit_transform(valid_train_df['all_labels'])
    
    print(f'Feature matrix shape : {X_df.shape}')
    print(f'Target matrix shape  : {y_matrix.shape}')
    print("Ready for training!")
else:
    print("\n" + "!"*50)
    print("❌ ERROR: CANNOT FIND PRE-EXTRACTED FEATURES!")
    print(f"Expected path: {INPUT_FEATURE_PATH}")
    print("Please make sure the dataset is attached to this Notebook.")
    print("!"*50 + "\n")
    raise RuntimeError("Dataset not found. Execution stopped to prevent training errors.")

Success! Loading features from: /kaggle/input/datasets/shuoyiyang/birdclef-2026-features-v2-r3-t3/features_v2_rating2_top3.parquet
Feature matrix shape : (60808, 196)
Target matrix shape  : (60808, 234)
Ready for training!


In [ ]:
X_matrix = X_df.drop(columns=['filename']).values
groups = X_df['filename'].values
y_matrix_full = y_matrix

gkf = GroupKFold(n_splits=5)


def run_cv(name, base_model):
    """Train `base_model` (wrapped in OvR) with GroupKFold and return mean AUC."""
    print(f"\nTesting model: {name} ...")
    fold_aucs = []
    model = OneVsRestClassifier(base_model, n_jobs=1)

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_matrix, y_matrix_full, groups)):
        X_train_fold, X_val_fold = X_matrix[train_idx], X_matrix[val_idx]
        y_train_fold, y_val_fold = y_matrix_full[train_idx], y_matrix_full[val_idx]

        model.fit(X_train_fold, y_train_fold)
        probs = model.predict_proba(X_val_fold)

        current_fold_scores = []
        for i in range(y_val_fold.shape[1]):
            if np.sum(y_val_fold[:, i]) > 0:
                score = roc_auc_score(y_val_fold[:, i], probs[:, i])
                current_fold_scores.append(score)

        avg_fold_score = np.mean(current_fold_scores) if current_fold_scores else 0.0
        fold_aucs.append(avg_fold_score)
        print(f"   Fold {fold+1}: AUC = {avg_fold_score:.4f} "
              f"(Evaluated on {len(current_fold_scores)} bird species)")

    avg_cv_score = float(np.mean(fold_aucs))
    print(f" {name} Final average CV Score: {avg_cv_score:.4f}")
    return avg_cv_score


model_comparison = {}

# LightGBM 
lgbm = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=8,
    num_leaves=63,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    n_jobs=1,
    random_state=42,
    verbose=-1,
    class_weight='balanced'
)
model_comparison["LightGBM"] = run_cv("LightGBM", lgbm)
print("\n--- LightGBM done. Partial leaderboard ---")
print(pd.Series(model_comparison).sort_values(ascending=False))


Testing model: LightGBM ...
   Fold 1: AUC = 0.8319 (Evaluated on 175 bird species)
   Fold 2: AUC = 0.8256 (Evaluated on 171 bird species)
   Fold 3: AUC = 0.8196 (Evaluated on 176 bird species)
   Fold 4: AUC = 0.8150 (Evaluated on 173 bird species)
   Fold 5: AUC = 0.8333 (Evaluated on 175 bird species)
 LightGBM Final average CV Score: 0.8251

--- LightGBM done. Partial leaderboard ---
LightGBM    0.825071
dtype: float64


In [ ]:
catboost = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=0,
    thread_count=-1,
    allow_writing_files=False,
)
model_comparison["CatBoost"] = run_cv("CatBoost", catboost)


Testing model: CatBoost ...
   Fold 1: AUC = 0.8376 (Evaluated on 175 bird species)
   Fold 2: AUC = 0.8356 (Evaluated on 171 bird species)
   Fold 3: AUC = 0.8353 (Evaluated on 176 bird species)
   Fold 4: AUC = 0.8312 (Evaluated on 173 bird species)
   Fold 5: AUC = 0.8366 (Evaluated on 175 bird species)
 CatBoost Final average CV Score: 0.8353


In [6]:
print("\n" + "="*35)
print("FINAL PERFORMANCE LEADERBOARD")
print(pd.Series(model_comparison).sort_values(ascending=False))
print("="*35)


FINAL PERFORMANCE LEADERBOARD
CatBoost    0.835261
LightGBM    0.825071
dtype: float64
